In [ ]:
import os, urllib.request
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

BASE = 'https://raw.githubusercontent.com/klaverhenrik/Deep-Learning-for-Visual-Recognition-2026/main/data/'
os.makedirs('data', exist_ok=True)
for fname in ['Cat03.jpg', 'paris.jpg', 'starry_night.jpg', 'cat_and_dog.jpg']:
    try:
        urllib.request.urlretrieve(BASE + fname, f'data/{fname}')
        print('downloaded', fname)
    except Exception as e:
        print(f'Could not download {fname} ({e}). Upload it manually into the data/ folder.')

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def load_image(path, size=224, normalise=True):
    """Load an image as a (1, 3, size, size) tensor, preprocessed for ImageNet models."""
    tfms = [T.Resize(size), T.CenterCrop(size), T.ToTensor()]
    if normalise:
        tfms.append(T.Normalize(IMAGENET_MEAN, IMAGENET_STD))
    return T.Compose(tfms)(Image.open(path).convert('RGB')).unsqueeze(0)

def denormalise(x):
    """Undo the ImageNet normalisation so a tensor can be shown with plt.imshow."""
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1)

# ── Visualising layer activations with forward hooks ──────────────────
weights = models.VGG16_Weights.IMAGENET1K_V1
model = models.vgg16(weights=weights)
model.eval()

# Make every ReLU a normal (not in-place) layer, so stored activations are never modified later
for m in model.modules():
    if isinstance(m, nn.ReLU):
        m.inplace = False

# Storage for captured activations
activations = {}

def make_hook(name):
    def hook(module, input, output):
        activations[name] = output.detach().cpu()
    return hook

# Hook the ReLU layers: a ReLU output is exactly 0 wherever a neuron is inactive,
# whereas a conv output (before the ReLU) is almost never exactly 0.
model.features[1].register_forward_hook(make_hook('relu1_1'))    # after conv1_1: 64 filters
model.features[29].register_forward_hook(make_hook('relu5_3'))   # after conv5_3: 512 filters

# Run a forward pass on a real image
img = load_image('data/Cat03.jpg')
with torch.no_grad():
    logits = model(img)
print('Prediction:', weights.meta['categories'][logits.argmax().item()])

# Inspect the captured activations
for name, act in activations.items():
    frac_zero       = (act == 0).float().mean().item()                        # zero activations
    frac_black_maps = (act.flatten(2).amax(dim=2) == 0).float().mean().item() # completely black maps
    print(f'{name}: shape={tuple(act.shape)}  zeros={frac_zero:.1%}  all-zero feature maps={frac_black_maps:.1%}')

# ── Visualising first-layer filters directly ──────────────────────────
# First conv layer weights have shape (out_ch, in_ch, H, W) = (64, 3, 3, 3)
conv1_weights = model.features[0].weight.data.cpu()

def normalise_filter(w):
    w = w - w.min()
    return w / (w.max() + 1e-8)

fmap1 = activations['relu1_1'][0]                       # (64, 224, 224)
fmap5 = activations['relu5_3'][0]                       # (512, 14, 14)
top5  = fmap5.flatten(1).amax(dim=1).argsort(descending=True)[:8]   # 8 most active maps

plt.figure(figsize=(3, 3)); plt.imshow(denormalise(img)[0].permute(1, 2, 0)); plt.axis('off'); plt.title('Input')

fig, axes = plt.subplots(3, 8, figsize=(16, 6.5))
for i in range(8):
    axes[0, i].imshow(normalise_filter(conv1_weights[i]).permute(1, 2, 0), interpolation='nearest')
    axes[1, i].imshow(fmap1[i], cmap='gray')
    axes[2, i].imshow(fmap5[top5[i]], cmap='gray')
for ax in axes.ravel():
    ax.axis('off')
axes[0, 0].set_title('conv1_1 filters (3x3 RGB)', loc='left')
axes[1, 0].set_title('relu1_1 feature maps (early: dense, edge/colour-like)', loc='left')
axes[2, 0].set_title('relu5_3: 8 most active of 512 maps (late: sparse, localised)', loc='left')
plt.show()

In [ ]:
import os, zipfile
import gdown
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Data: the same cats / dogs / horses dataset as Lab 2 ─────────────
if not os.path.isdir('data/cat'):
    file_id = '1KDMC39ba1MAL83FLLoSVSJY2KOmFR1aj'
    gdown.download(f'https://drive.google.com/uc?id={file_id}', 'data.zip', quiet=False)
    with zipfile.ZipFile('data.zip', 'r') as z:
        z.extractall('.')

preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
dataset = ImageFolder('data', transform=preprocess)   # one sub-folder per class
loader  = DataLoader(dataset, batch_size=32, shuffle=False)
print('Classes:', dataset.classes, '| number of images:', len(dataset))

# ── Extract embeddings from the second FC layer (4096-d) ──────────────
model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# VGG16 classifier: [Linear(25088,4096), ReLU, Dropout,
#                    Linear(4096,4096),  ReLU, Dropout,
#                    Linear(4096,1000)]
# We want the 4096-d output of the second FC layer (after its ReLU)
embedding_model = nn.Sequential(
    model.features,
    model.avgpool,
    nn.Flatten(),
    model.classifier[:5],   # up to and including the second ReLU
).to(device).eval()

chunks = []
with torch.no_grad():
    for images, _ in loader:
        chunks.append(embedding_model(images.to(device)).cpu())
embeddings = torch.cat(chunks).numpy()              # (N, 4096)
labels     = np.array(dataset.targets)              # (N,)
print(f'Embeddings shape: {embeddings.shape}')

# ── k-NN in feature space ─────────────────────────────────────────────
knn = NearestNeighbors(n_neighbors=6, metric='cosine').fit(embeddings)
dists, idxs = knn.kneighbors(embeddings)
neighbours  = idxs[:, 1:]                           # drop each image itself (nearest neighbour #0)

same_class = (labels[neighbours] == labels[:, None]).mean()
print(f'Fraction of 5-NN neighbours with the same class as the query: {same_class:.1%}')

# Show one query image and its 5 nearest neighbours
q = int(np.random.default_rng(0).integers(len(dataset)))
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for ax, i, title in zip(axes, [q] + list(neighbours[q]), ['query'] + [f'NN {k}' for k in range(1, 6)]):
    ax.imshow(Image.open(dataset.samples[i][0]).convert('RGB'))
    ax.set_title(f'{title}: {dataset.classes[labels[i]]}')
    ax.axis('off')
plt.show()

# ── t-SNE dimensionality reduction ────────────────────────────────────
# Step 1: reduce to 50D with PCA (t-SNE is slow on raw 4096-d data)
pca     = PCA(n_components=min(50, len(embeddings) - 1))
emb_pca = pca.fit_transform(embeddings)
print(f'PCA variance explained: {pca.explained_variance_ratio_.sum():.1%}')

# Step 2: t-SNE to 2D (perplexity must be smaller than the number of images)
tsne   = TSNE(n_components=2, perplexity=min(30, len(embeddings) // 4), random_state=0)
emb_2d = tsne.fit_transform(emb_pca)                # (N, 2)

plt.figure(figsize=(6, 6))
for cls, name in enumerate(dataset.classes):
    mask = labels == cls
    plt.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=name, s=12)
plt.legend(); plt.title('t-SNE of VGG16 FC7 features')
plt.show()
# Good result: tight clusters per class, with the classes clearly separated

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def load_image(path, size=224, normalise=True):
    """Load an image as a (1, 3, size, size) tensor, preprocessed for ImageNet models."""
    tfms = [T.Resize(size), T.CenterCrop(size), T.ToTensor()]
    if normalise:
        tfms.append(T.Normalize(IMAGENET_MEAN, IMAGENET_STD))
    return T.Compose(tfms)(Image.open(path).convert('RGB')).unsqueeze(0)

def denormalise(x):
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1)

# ════════════════════════════════════════════════════════════════════
# GRAD-CAM: Class Activation Mapping via gradients
# Works with any CNN that has a convolutional layer before pooling.
# ════════════════════════════════════════════════════════════════════

class GradCAM:
    def __init__(self, model, target_layer):
        self.model        = model
        self.feature_maps = None   # stores forward activations
        self.gradients    = None   # stores backward gradients

        # Forward hook: capture the feature maps of the target layer
        target_layer.register_forward_hook(self._save_features)

    def _save_features(self, module, input, output):
        self.feature_maps = output.detach()           # (1, C, H, W)
        # Ask autograd to hand us d(score)/d(output) during the backward pass
        if output.requires_grad:
            output.register_hook(self._save_gradients)

    def _save_gradients(self, grad):
        self.gradients = grad.detach()                # (1, C, H, W)

    def generate(self, x, class_idx=None):
        self.model.eval()
        logits = self.model(x)     # forward pass: triggers _save_features

        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        # Backpropagate the raw logit of the target class (not the softmax probability)
        self.model.zero_grad()
        logits[0, class_idx].backward()   # fills self.gradients

        # Step 1: global-average-pool the gradients over the spatial dims
        # weights: how important is each channel for this class?
        weights = self.gradients.mean(dim=[2, 3], keepdim=True)       # (1, C, 1, 1)

        # Step 2: weighted sum of the feature maps
        cam = (weights * self.feature_maps).sum(dim=1, keepdim=True)  # (1, 1, H, W)

        # Step 3: ReLU: keep only regions that support the class
        cam = F.relu(cam)

        # Step 4: upsample to the input image size
        cam = F.interpolate(cam, size=x.shape[2:], mode='bilinear', align_corners=False)

        # Normalise to [0, 1]
        cam = cam.squeeze()                                           # (H, W)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, class_idx

# ── Set up VGG16 ──────────────────────────────────────────────────────
weights_vgg = models.VGG16_Weights.IMAGENET1K_V1
categories  = weights_vgg.meta['categories']
model = models.vgg16(weights=weights_vgg).eval()
for m in model.modules():
    if isinstance(m, nn.ReLU):
        m.inplace = False            # hooks and in-place ReLUs do not mix

# Target: the ReLU after the last conv layer (features[29] in VGG16)
grad_cam = GradCAM(model, target_layer=model.features[29])

# ── One image, two different classes ──────────────────────────────────
x = load_image('data/cat_and_dog.jpg')

with torch.no_grad():
    logits = model(x)[0]
cat_idx = 281 + logits[281:286].argmax().item()    # best-scoring cat class (281-285)
dog_idx = 151 + logits[151:269].argmax().item()    # best-scoring dog class (151-268)
top_idx = logits.argmax().item()                   # the model's overall prediction
print('Top prediction:', categories[top_idx])

original = denormalise(x)[0].permute(1, 2, 0)
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(original); axes[0].set_title('Input')
for ax, idx in zip(axes[1:], [cat_idx, dog_idx]):
    cam, _ = grad_cam.generate(x, class_idx=idx)
    ax.imshow(original); ax.imshow(cam.numpy(), alpha=0.5, cmap='jet')
    ax.set_title(f'Grad-CAM for: {categories[idx]}')
for ax in axes: ax.axis('off')
plt.show()
# Expected: the "cat" map lights up the cat, the "dog" map lights up the dog.
# Same image, same network: only the class whose score we backpropagate differs.

# ── Simple gradient saliency map (pixel-level), for comparison ────────
x2 = x.clone().requires_grad_(True)
logits2 = model(x2)
cls2    = logits2.argmax(1).item()
model.zero_grad()
logits2[0, cls2].backward()

saliency = x2.grad.abs().max(dim=1)[0]            # max over colour channels: (1, 224, 224)

plt.figure(figsize=(5, 5))
plt.imshow(saliency[0].numpy(), cmap='gray'); plt.axis('off')
plt.title(f'Gradient saliency: {categories[cls2]}')
plt.show()

In [ ]:
import torch
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def load_image(path, size=224, normalise=True):
    tfms = [T.Resize(size), T.CenterCrop(size), T.ToTensor()]
    if normalise:
        tfms.append(T.Normalize(IMAGENET_MEAN, IMAGENET_STD))
    return T.Compose(tfms)(Image.open(path).convert('RGB')).unsqueeze(0)

def denormalise(x):
    mean = torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1)
    std  = torch.tensor(IMAGENET_STD).view(1, 3, 1, 1)
    return (x.cpu() * std + mean).clamp(0, 1)

def frozen_front(model, layer_idx):
    """The first layers of VGG up to and including model.features[layer_idx], frozen."""
    feat = model.features[:layer_idx + 1].to(device).eval()
    for p in feat.parameters():
        p.requires_grad_(False)       # freeze the network
    return feat

# ── Content reconstruction: invert a layer's activations ─────────────
def content_reconstruction(model, content_img, layer_idx, n_steps=300, lr=0.05):
    """
    Find an image whose activations at layer_idx match those of content_img.
    model:       pretrained VGG
    content_img: (1, 3, H, W) tensor (the reference image)
    layer_idx:   which VGG features layer to match
    """
    feat = frozen_front(model, layer_idx)
    content_img = content_img.to(device)

    # Target activations of the content image
    with torch.no_grad():
        target = feat(content_img)

    # Initialise the generated image with small random noise
    generated = (torch.randn_like(content_img) * 0.01).requires_grad_(True)
    optimiser = torch.optim.Adam([generated], lr=lr)

    for step in range(n_steps):
        optimiser.zero_grad()
        loss = F.mse_loss(feat(generated), target)   # match activations
        loss.backward()                               # gradients flow to the pixels
        optimiser.step()

        with torch.no_grad():
            generated.clamp_(-2.5, 2.5)               # keep pixels in a valid range

        if step % 100 == 0:
            print(f'  layer {layer_idx:2d}  step {step:4d}  loss: {loss.item():.4f}')

    return generated.detach().cpu()

# ── Filter visualisation via gradient ascent ──────────────────────────
def visualise_filter(model, layer_idx, filter_idx, n_steps=150, lr=0.1):
    """
    Find the input image that maximally activates a specific filter.
    Gradient ASCENT on the mean filter activation.
    """
    feat = frozen_front(model, layer_idx)

    # Start from small random noise
    img = (torch.randn(1, 3, 224, 224, device=device) * 0.01).requires_grad_(True)
    optimiser = torch.optim.Adam([img], lr=lr)

    for step in range(n_steps):
        optimiser.zero_grad()
        # Loss = NEGATIVE mean activation of the target filter
        # (negative because the optimiser minimises, but we want to maximise)
        loss = -feat(img)[0, filter_idx].mean()
        loss.backward()
        optimiser.step()

        # Regularise: keep the image natural-looking
        with torch.no_grad():
            img.clamp_(-2.5, 2.5)
            if step % 10 == 9:                       # small blur suppresses high-frequency noise
                img.copy_(TF.gaussian_blur(img, kernel_size=3, sigma=0.5))

    return img.detach().cpu()

def to_display(x):
    """Scale an optimised image to [0, 1] for plt.imshow."""
    x = x[0]
    x = (x - x.min()) / (x.max() - x.min() + 1e-8)
    return x.permute(1, 2, 0)

model = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1)

# ── 1) Reconstruct a real image from different layers ─────────────────
content = load_image('data/Cat03.jpg')
layers  = {3: 'relu1_2', 8: 'relu2_2', 15: 'relu3_3', 22: 'relu4_3', 29: 'relu5_3'}

fig, axes = plt.subplots(1, 1 + len(layers), figsize=(3 * (1 + len(layers)), 3.4))
axes[0].imshow(denormalise(content)[0].permute(1, 2, 0)); axes[0].set_title('Original')
for ax, (idx, name) in zip(axes[1:], layers.items()):
    recon = content_reconstruction(model, content, layer_idx=idx)
    ax.imshow(denormalise(recon)[0].permute(1, 2, 0)); ax.set_title(f'from {name}')
for ax in axes: ax.axis('off')
plt.show()
# Expected: early layers reproduce the image almost exactly; deeper layers keep the
# overall layout/semantics but lose fine detail and colour precision.

# ── 2) Visualise what individual filters respond to ───────────────────
# Layer indices are conv layers: 0 = conv1_1, 14 = conv3_3, 28 = conv5_3
filters_to_show = {0: [0, 1, 2, 3], 14: [0, 1, 2, 3], 28: [0, 1, 2, 3]}
fig, axes = plt.subplots(len(filters_to_show), 4, figsize=(10, 7.5))
for r, (layer_idx, fids) in enumerate(filters_to_show.items()):
    for c, f in enumerate(fids):
        vis = visualise_filter(model, layer_idx=layer_idx, filter_idx=f)
        axes[r, c].imshow(to_display(vis)); axes[r, c].axis('off')
        axes[r, c].set_title(f'features[{layer_idx}] filter {f}')
plt.show()
# Expected: early filters: oriented edges / colour blobs; middle: textures; late: object-part-like patterns

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def load_image_01(path, size=256):
    """Load an image as a (1, 3, size, size) tensor with pixels in [0, 1]."""
    tfm = T.Compose([T.Resize(size), T.CenterCrop(size), T.ToTensor()])
    return tfm(Image.open(path).convert('RGB')).unsqueeze(0)

# ════════════════════════════════════════════════════════════════════
# NEURAL STYLE TRANSFER — full implementation
# Content + Style loss via a frozen VGG19 network.
# ════════════════════════════════════════════════════════════════════

def gram_matrix(feature_map):
    """
    Compute the Gram matrix of a feature map.
    feature_map: (1, C, H, W)
    Returns:     (C, C) matrix of feature co-occurrences
    """
    B, C, H, W = feature_map.shape
    feats = feature_map.reshape(C, H * W)           # (C, H*W)
    # Gram matrix = feats @ feats^T, normalised by the number of elements
    return feats @ feats.t() / (C * H * W)

class StyleTransfer:
    def __init__(self,
                 content_layers=['features.22'],           # relu4_2 in VGG19
                 style_layers=['features.1', 'features.6',   # relu1_1, relu2_1
                               'features.11', 'features.20', # relu3_1, relu4_1
                               'features.29']):             # relu5_1
        self.content_layers = content_layers
        self.style_layers   = style_layers
        self.last_layer     = max(int(k.split('.')[1]) for k in content_layers + style_layers)

        # Load VGG19 and replace max pooling by average pooling (smoother gradients)
        vgg = models.vgg19(weights=models.VGG19_Weights.IMAGENET1K_V1)
        for i, layer in enumerate(vgg.features):
            if isinstance(layer, nn.MaxPool2d):
                vgg.features[i] = nn.AvgPool2d(2, 2)
            elif isinstance(layer, nn.ReLU):
                layer.inplace = False

        self.model = vgg.features.to(device).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)   # freeze all weights

        # VGG expects ImageNet-normalised input; we keep our images in [0, 1] and normalise here
        self.normalise = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

    def get_features(self, img):
        """Run a forward pass and collect activations at the chosen layers."""
        features = {}
        x = self.normalise(img)
        for i, layer in enumerate(self.model):
            x = layer(x)
            key = f'features.{i}'
            if key in self.content_layers or key in self.style_layers:
                features[key] = x
            if i == self.last_layer:
                break                  # no need to run the deeper layers
        return features

    def run(self, content_img, style_img, n_steps=300,
            content_weight=1.0, style_weight=1e6):
        content_img, style_img = content_img.to(device), style_img.to(device)

        # Pre-compute target features (only once — images are fixed)
        content_feats = self.get_features(content_img)
        style_feats   = self.get_features(style_img)
        style_grams   = {k: gram_matrix(v) for k, v in style_feats.items()
                         if k in self.style_layers}

        # Initialise the generated image from the content image (better convergence than noise)
        generated = content_img.clone().requires_grad_(True)
        optimiser = torch.optim.LBFGS([generated], max_iter=20)

        step = [0]
        def closure():
            generated.data.clamp_(0, 1)
            optimiser.zero_grad()
            gen_feats = self.get_features(generated)

            # Content loss: MSE between feature maps at the content layer
            L_content = sum(
                F.mse_loss(gen_feats[l], content_feats[l])
                for l in self.content_layers
            )

            # Style loss: MSE between Gram matrices at the style layers
            L_style = sum(
                F.mse_loss(gram_matrix(gen_feats[l]), style_grams[l])
                for l in self.style_layers
            )

            loss = content_weight * L_content + style_weight * L_style
            loss.backward()

            step[0] += 1
            if step[0] % 50 == 0:
                print(f'Step {step[0]:4d}  content: {L_content.item():.2f}',
                      f' style: {L_style.item():.4f}')
            return loss

        for _ in range(n_steps // 20 + 1):
            optimiser.step(closure)

        return generated.detach().clamp(0, 1).cpu()

# ── Usage: a photo of Paris in the style of Starry Night ──────────────
content = load_image_01('data/paris.jpg')
style   = load_image_01('data/starry_night.jpg')

nst    = StyleTransfer()
result = nst.run(content, style, n_steps=300, content_weight=1.0, style_weight=1e6)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, im, title in zip(axes, [content, style, result], ['Content', 'Style', 'Result']):
    ax.imshow(im[0].permute(1, 2, 0)); ax.set_title(title); ax.axis('off')
plt.show()

print('Content weight high: preserves photo, adds subtle style')
print('Style weight high:   strong artistic style, loses photo details')
print('Content layer: relu4_2 (middle layers work best)')
print('Style layers:  relu1_1 through relu5_1 (all scales of texture)')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
import matplotlib.pyplot as plt
from PIL import Image

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

def load_image(path, size=224, normalise=True):
    tfms = [T.Resize(size), T.CenterCrop(size), T.ToTensor()]
    if normalise:
        tfms.append(T.Normalize(IMAGENET_MEAN, IMAGENET_STD))
    return T.Compose(tfms)(Image.open(path).convert('RGB')).unsqueeze(0)

# ── Fast Gradient Sign Method (FGSM) adversarial attack ──────────────
def fgsm_attack(model, image, true_label, epsilon=0.03):
    """
    Untargeted attack: make the model misclassify the image.
    image:   (1, 3, H, W) with pixel values in [0, 1]
    epsilon: perturbation magnitude (small = imperceptible to humans)
    """
    model.eval()
    img = image.clone().requires_grad_(True)

    # Forward pass
    logits = model(img)
    loss   = F.cross_entropy(logits, true_label)

    # Backpropagate to the input image (not to model weights)
    model.zero_grad()
    loss.backward()

    # FGSM: perturb in the direction that increases the loss
    # sign() makes the perturbation exactly +/- epsilon per pixel
    perturbation    = epsilon * img.grad.sign()
    adversarial_img = (img + perturbation).clamp(0, 1).detach()

    return adversarial_img

def targeted_attack(model, image, target_class, epsilon=0.03, n_steps=50):
    """
    Targeted attack: make the model output a specific class.
    Uses iterative gradient ascent on the target class score.
    """
    model.eval()
    img = image.clone().requires_grad_(True)
    opt = torch.optim.Adam([img], lr=epsilon)

    for step in range(n_steps):
        opt.zero_grad()
        logits = model(img)
        # Maximise the target class score (gradient ascent → negate for Adam)
        loss = -logits[0, target_class]
        loss.backward()
        opt.step()

        # Project back: keep perturbation small and pixels valid
        with torch.no_grad():
            delta = img - image
            delta.clamp_(-epsilon, epsilon)  # L-inf constraint
            img.data = (image + delta).clamp(0, 1)

    return img.detach()

# ── Test ──────────────────────────────────────────────────────────────
weights = models.ResNet50_Weights.IMAGENET1K_V2
categories = weights.meta['categories']

# Normalisation is part of the model, so the attack works directly on pixels in [0, 1]
model = nn.Sequential(T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
                      models.resnet50(weights=weights)).eval()
for p in model.parameters():
    p.requires_grad_(False)

image = load_image('data/Cat03.jpg', normalise=False)   # real image, pixels in [0, 1]

def predict(x):
    probs = F.softmax(model(x), dim=1)
    conf, idx = probs.max(dim=1)
    return idx.item(), conf.item()

# Attack the model's own prediction on the clean image
clean_idx, clean_conf = predict(image)
print(f'Clean image:       {categories[clean_idx]} ({clean_conf:.1%})')

# FGSM attack
eps       = 0.03
adv_img   = fgsm_attack(model, image, torch.tensor([clean_idx]), epsilon=eps)
perturbation_magnitude = (adv_img - image).abs().max().item()
print(f'Max pixel perturbation: {perturbation_magnitude:.4f} (eps={eps}) — barely visible')

adv_idx, adv_conf = predict(adv_img)
print(f'Adversarial image: {categories[adv_idx]} ({adv_conf:.1%})')

# Targeted attack: make the model say "goldfish" (ImageNet class 1)
target_class = 1
tgt_img = targeted_attack(model, image, target_class, epsilon=eps)
tgt_idx, tgt_conf = predict(tgt_img)
print(f'Targeted attack:   {categories[tgt_idx]} ({tgt_conf:.1%})   [target was: {categories[target_class]}]')

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
panels = [(image, f'Clean: {categories[clean_idx]}'),
          (adv_img, f'FGSM: {categories[adv_idx]}'),
          ((adv_img - image) / (2 * eps) + 0.5, 'FGSM perturbation (amplified)'),
          (tgt_img, f'Targeted: {categories[tgt_idx]}')]
for ax, (im, title) in zip(axes, panels):
    ax.imshow(im[0].clamp(0, 1).permute(1, 2, 0)); ax.set_title(title); ax.axis('off')
plt.show()